# Mosaic — epitope-targeted binder design on 5G51

Gradient-based **multi-objective** design ([escalante-bio/mosaic](https://github.com/escalante-bio/mosaic)).
Unlike BindCraft, which designs then filters, this puts solubility and stability **into the objective**.

### Targets

| group | PDB residues | `epitope_idx` | notes |
|---|---|---|---|
| **A_ridge** | Q325, P326, V384, N386 | `[65, 66, 124, 126]` | one face, all at zero in-virion occlusion |
| **B_triad** | H277, D294 | `[17, 34]` | opposite face; triad refuted, lowest BepiPred |

`epitope_idx = PDB − 260`, asserted at runtime.

### Why the install cell looks unusual — read this once

`pyproject.toml` routes **13 dependencies through `[tool.uv.sources]`**, which only `uv` reads.
Plain `pip install -e .` silently resolves those names against **PyPI instead**, and three of them
collide with unrelated packages — `protenix`, `boltz` and `opendde` on PyPI are *not* Escalante's.
The result is an install that appears to succeed while importing the wrong code, which is exactly what
happened on the earlier attempt (`pytorch-lightning -> boltz -> mosaic` in the log is PyPI's boltz).

So cell 2 installs all 13 explicitly by git URL with `--force-reinstall`, then puts `src/` on the path
rather than building the package. Mosaic is pure Python and ships its weights in the repo, so there is
nothing to compile.


## 1 · Environment

In [ ]:
import sys, subprocess
print("python    :", sys.version.split()[0])
print("executable:", sys.executable)
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"])
assert sys.version_info[:2] == (3,12), f"mosaic pins python==3.12.*, got {sys.version_info[:2]}"
print("\nOK")

## 2 · Install — the fixed cell

Takes 10–20 min on first run. Output streams live so nothing is hidden.

In [ ]:
import os, sys, subprocess, importlib

BASE = next(p for p in ("/content", "/kaggle/working", os.path.expanduser("~"))
            if os.path.isdir(p) and os.access(p, os.W_OK))
REPO = f"{BASE}/mosaic_esc"
SRC  = f"{REPO}/src"

def run(cmd):
    """Stream output live - captured output disappears if anything raises early."""
    print("$", " ".join(cmd[:6]), "..." if len(cmd) > 6 else "", flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail.append(line)
        if any(k in line for k in ("ERROR","error:","Building","Successfully","Cloning","Resolved")):
            print("   ", line.rstrip()[:200], flush=True)
    rc = p.wait()
    if rc:
        print("!! FAILED rc =", rc)
        print("".join(tail[-40:]))
    return rc

if not os.path.isdir(REPO + "/.git"):
    run(["git","clone","--depth","1","https://github.com/escalante-bio/mosaic", REPO])
assert os.path.isdir(SRC + "/mosaic"), f"clone failed: no {SRC}/mosaic"
print("repo:", REPO)

# --- the 13 git-sourced deps, installed ONE AT A TIME ---
# A single pip command with all 13 is atomic: one failure installs NONE of them.
# Installed individually WITH their dependencies, so each reports its own status.
GIT_DEPS = [
    ("joltz",              "git+https://github.com/nboyd/joltz"),
    ("protenix",           "git+https://github.com/escalante-bio/protenij"),
    ("jablang",            "git+https://github.com/escalante-bio/jablang"),
    ("joltzgen",           "git+https://github.com/escalante-bio/joltzgen.git"),
    ("jopenfold3",         "git+https://github.com/escalante-bio/jopenfold3.git"),
    ("jproteina_complexa", "git+https://github.com/escalante-bio/jproteina-complexa.git"),
    ("boltz",              "git+https://github.com/escalante-bio/boltz.git"),
    ("boltzgen",           "git+https://github.com/escalante-bio/boltzgen.git"),
    ("esmjfold2",          "git+https://github.com/escalante-bio/esmjfold2.git"),
    ("jpromera",           "git+https://github.com/escalante-bio/jpromera.git"),
    ("jopendde",           "git+https://github.com/escalante-bio/jopendde.git"),
    ("opendde",            "git+https://github.com/escalante-bio/jopendde.git#subdirectory=OpenDDE"),
    ("esm",                "git+https://github.com/Biohub/esm.git"),
]
print("\n=== git-sourced dependencies, one at a time ===")
failed = []
for name, url in GIT_DEPS:
    rc = run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", url])
    ok = "OK  " if rc == 0 else "FAIL"
    if rc: failed.append(name)
    print(f"  {ok} {name}", flush=True)

print("\n=== remaining PyPI deps ===")
run([sys.executable,"-m","pip","install","-q",
     "dm-haiku>=0.0.13","esm2quinox>=0.1.0","flax>=0.10.2","gemmi>=0.6.5",
     "ml-collections>=1.0.0","optax>=0.2.4","equinox>=0.13.0","ablang>=0.3.1",
     "transformers","accelerate","safetensors","polars","httpx","scipy>=1.15.1",
     "matplotlib","numpy>=2.1","pytorch-lightning","einops","hydra-core","omegaconf"])

if failed:
    print("\n!! these failed to install:", failed)
    print("!! retrying them individually with full output so the real error is visible")
    for name, url in GIT_DEPS:
        if name in failed:
            print(f"\n----- {name} -----")
            r = subprocess.run([sys.executable,"-m","pip","install","--force-reinstall",url],
                               capture_output=True, text=True)
            print(r.stdout[-2500:]); print(r.stderr[-2500:])

# --- import via path, no build step ---
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

import mosaic
print("\nmosaic ->", mosaic.__file__)
from mosaic.models.boltz2 import Boltz2
from mosaic.structure_prediction import TargetChain
import mosaic.losses.structure_prediction as sp
from mosaic.losses.protein_mpnn import InverseFoldingSequenceRecovery
from mosaic.proteinmpnn.mpnn import load_mpnn_sol
from mosaic.losses.trigram import TrigramLL
from mosaic.optimizers import simplex_APGM

# Second predictor is OPTIONAL - a protenix failure must not block the whole run.
HAVE_PROTENIX = False
try:
    from mosaic.models.protenix import ProtenixV2
    HAVE_PROTENIX = True
    print("ProtenixV2 available (dual-predictor mode)")
except Exception as e:
    print("ProtenixV2 NOT available -> Boltz2-only mode. Reason:", repr(e)[:140])

print("ALL REQUIRED MOSAIC IMPORTS OK")

import jax
print("jax:", jax.__version__, "| devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "No GPU visible to JAX"
print("\nPREFLIGHT PASSED")

### 2b · If cell 2 failed

Run this, then re-run cell 2. It reports which import breaks and where each package came from —
a package resolving to `site-packages` rather than a git checkout means PyPI won the race.

In [ ]:
import importlib, sys
sys.path.insert(0, SRC) if 'SRC' in dir() and SRC not in sys.path else None
for m in ["joltz","protenix","boltz","boltzgen","jablang","joltzgen","jopenfold3",
          "jproteina_complexa","jpromera","jopendde","opendde","esmjfold2","esm",
          "esm2quinox","equinox","jax","mosaic"]:
    try:
        mod = importlib.import_module(m)
        print(f"  OK   {m:<20} {getattr(mod,'__file__','(ns)')}")
    except Exception as e:
        print(f"  FAIL {m:<20} {type(e).__name__}: {str(e)[:90]}")

## 3 · Target and epitopes

In [ ]:
TARGET_SEQ = (
    "EEYRAKTGYAPYYAGVWHSFNNSNSLVFRWGSASDQIAQWPTISVPRGELAFLRIKDGKQ"
    "AAVGTQPWRTMVVWPSGHGYNIGIPTYNAERARQLAQHLYGGGSLTDEKAKQLFVPANQQ"
    "GPGKVSNGNPVWEVMRAPLATQRAHIQDFEFIEAIPE"
)
OFFSET = 260
assert len(TARGET_SEQ) == 157

GROUPS = {
    "A_ridge": {"pdb": [325, 326, 384, 386], "expect": "QPVN"},
    "B_triad": {"pdb": [277, 294],           "expect": "HD"},
}
for name, g in GROUPS.items():
    g["idx"] = [p - OFFSET for p in g["pdb"]]
    got = "".join(TARGET_SEQ[i] for i in g["idx"])
    print(f"{name}: PDB {g['pdb']} -> idx {g['idx']} -> '{got}'")
    assert got == g["expect"], f"index mapping wrong for {name}"
print("\nindex mapping verified")

RUN_GROUPS    = ["A_ridge", "B_triad"]
BINDER_LENGTH = 80
print("groups:", RUN_GROUPS, "| binder_length:", BINDER_LENGTH)

## 4 · Models

| object | role |
|---|---|
| `Boltz2` | complex structure prediction — drives interface losses |
| `ProtenixV2` | **second, independent** structure predictor — same interface objective |
| `load_mpnn_sol(0.05)` | **SolubleMPNN** — the solubility term |
| `StabilityModel` | folding stability |
| `ESM2PseudoLikelihood` | sequence naturalness → expression |
| `TrigramLL` | cheap naturalness prior |

Weights download from HuggingFace on first use.

In [ ]:
import jax, numpy as np

# --- TWO independent structure predictors ---
# Boltz2 (MIT lineage) and ProtenixV2 (ByteDance AF3 reproduction) are separate
# codebases with separate training. A design that satisfies BOTH is far less likely to
# be exploiting one model's blind spot - the single biggest failure mode of
# hallucination-based design. Boltz1 would NOT give this: it shares Boltz2's lineage.
TWO_MODELS = True and HAVE_PROTENIX     # auto-off if protenix failed to install

model  = Boltz2();                    print("Boltz2 loaded")
model2 = None
if TWO_MODELS:
    try:
        model2 = ProtenixV2();        print("ProtenixV2 loaded")
    except Exception as e:
        print("Protenix unavailable, falling back to single model:", repr(e)[:160])
        TWO_MODELS = False

mpnn_sol = load_mpnn_sol(0.05);       print("SolubleMPNN loaded")
trigram_ll = TrigramLL.from_pkl();    print("trigram loaded")

USE_ESM = True
esm_pll = stability_loss = None
if USE_ESM:
    try:
        import esm, esm2quinox
        from mosaic.losses.esm import ESM2PseudoLikelihood
        from mosaic.losses.stability import StabilityModel
        tm, _ = esm.pretrained.esm2_t33_650M_UR50D()
        esm_q = esm2quinox.from_torch(tm)
        esm_pll = ESM2PseudoLikelihood(esm_q)
        stability_loss = StabilityModel.from_pretrained(esm_q)
        print("ESM-2 + stability loaded")
    except Exception as e:
        print("ESM/stability unavailable, continuing without:", repr(e)[:160])
        USE_ESM = False

## 5 · Features

Epitope-independent, so built once and reused by both groups.

In [ ]:
complex_features, complex_writer = model.binder_features(
    binder_length=BINDER_LENGTH, chains=[TargetChain(sequence=TARGET_SEQ)])
mono_features, _ = model.binder_features(binder_length=BINDER_LENGTH, chains=[])
print("Boltz2 features built")

# each model has its own featurizer - they are NOT interchangeable
complex_features2 = mono_features2 = None
if TWO_MODELS:
    complex_features2, _ = model2.binder_features(
        binder_length=BINDER_LENGTH, chains=[TargetChain(sequence=TARGET_SEQ)])
    print("ProtenixV2 features built")

## 6 · Objective

**Interface** — `BinderTargetContact(epitope_idx=...)` is the epitope constraint. `IPSAE_min` is
weighted heavily because ipSAE predicts binding (AUC ≈ 0.81) where ipTM does not (≈ 0.5).

**Solubility / expression** — `InverseFoldingSequenceRecovery(mpnn_sol)` pushes the sequence toward
SolubleMPNN's preference, attacking the surface-hydrophobicity failure that killed 56% of the
BindCraft trajectories. Plus stability and ESM likelihood.

**Foldability** — the monomer block, so the binder folds unbound.

**Two predictors.** The interface objective is evaluated by **Boltz2 and ProtenixV2** and both are
summed, so a design has to satisfy two independently-trained models. This is the main defence against
adversarial designs — sequences that score beautifully on one predictor because they exploit its
blind spot, then fail everywhere else. Set `TWO_MODELS = False` to halve time and VRAM.

In [ ]:
W_EPITOPE, W_IPSAE, W_PTM_E = 4.0, 2.0, 1.0
W_SOLUBLE, W_STABILITY, W_ESM, W_MONOMER = 5.0, 0.1, 0.5, 0.5
W_MODEL2 = 1.0        # weight on the second predictor (Protenix). 1.0 = equal footing.

def build_objective(epitope_idx):
    interface = (
        W_EPITOPE * sp.BinderTargetContact(epitope_idx=epitope_idx)
        + W_IPSAE * sp.IPSAE_min()
        + W_PTM_E * sp.pTMEnergy()
        + sp.WithinBinderContact()
        + sp.DistogramRadiusOfGyration(target_radius=15.0)
        + 0.3 * sp.HelixLoss()
        + W_SOLUBLE * InverseFoldingSequenceRecovery(mpnn_sol, temp=jax.numpy.array(0.01))
    )
    L = model.build_loss(loss=interface, features=complex_features, recycling_steps=1)

    # second predictor on the SAME interface objective - the design must satisfy both
    if TWO_MODELS:
        interface2 = (
            W_EPITOPE * sp.BinderTargetContact(epitope_idx=epitope_idx)
            + W_IPSAE * sp.IPSAE_min()
            + sp.WithinBinderContact()
            + sp.DistogramRadiusOfGyration(target_radius=15.0)
        )
        L = L + W_MODEL2 * model2.build_loss(
            loss=interface2, features=complex_features2, recycling_steps=2)

    L = L + W_MONOMER * model.build_loss(
        loss=0.2 * sp.PLDDTLoss() + sp.DistogramRadiusOfGyration(target_radius=15.0)
             + 0.3 * sp.HelixLoss(),
        features=mono_features, recycling_steps=1)
    L = L + trigram_ll
    if USE_ESM:
        L = L + W_ESM * esm_pll + W_STABILITY * stability_loss
    return L

print("objective builder ready")
print("  predictors      :", "Boltz2 + ProtenixV2" if TWO_MODELS else "Boltz2 only")
print("  SolubleMPNN     : weight", W_SOLUBLE)
print("  ESM + stability :", "on" if USE_ESM else "OFF")

## 7 · Run

Every design is fsync'd to Drive the moment it finishes, so an interrupt costs at most the in-flight
design. Re-running resumes per group.

In [ ]:
import numpy as np, jax, json, os, datetime, time, traceback

REQUIRE_DRIVE = True
ROOT = "/content/drive/MyDrive/Mosaic"
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print("Drive mount raised:", repr(e)[:150])

if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(ROOT, exist_ok=True)
    _p = f"{ROOT}/.writetest"
    with open(_p,"w") as fh: fh.write("ok"); fh.flush(); os.fsync(fh.fileno())
    os.remove(_p); print("Drive OK ->", ROOT)
elif REQUIRE_DRIVE:
    raise RuntimeError("Drive not mounted - designs would be lost. Complete the auth "
                       "popup and re-run, or set REQUIRE_DRIVE=False to accept the risk.")
else:
    ROOT = f"{BASE}/mosaic_out"; print("!! EPHEMERAL disk:", ROOT)

N_DESIGNS, N_STEPS, CU_PER_HOUR = 30, 150, 9
TOKENS = "ACDEFGHIKLMNPQRSTVWY"
TIMES  = []

def _atomic(path, text):
    """Write + flush + fsync so the file is really on Drive before we continue."""
    with open(path, "w") as fh:
        fh.write(text); fh.flush(); os.fsync(fh.fileno())

def write_status(**kw):
    """Live heartbeat at the campaign root - readable from another tab mid-run."""
    kw["updated"] = datetime.datetime.now().isoformat()
    _atomic(f"{ROOT}/status.json", json.dumps(kw, indent=2))
    _atomic(f"{ROOT}/status.txt",
            " | ".join(f"{k}={v}" for k, v in kw.items() if k != "updated")
            + f"\nupdated {kw['updated']}\n")

def save_design(out_dir, rec, pssm):
    os.makedirs(out_dir, exist_ok=True)
    jl = f"{out_dir}/designs.jsonl"
    with open(jl,"a") as fh:
        fh.write(json.dumps(rec)+"\n"); fh.flush(); os.fsync(fh.fileno())
    np.save(f"{out_dir}/pssm_{rec['group']}_{rec['seed']}.npy", np.array(pssm))

    recs = [json.loads(l) for l in open(jl)]
    # FASTA - always an orderable file on disk, rebuilt every design
    _atomic(f"{out_dir}/designs.fasta",
            "".join(f">{r['group']}_d{r['design']}_s{r['seed']}_len{r['length']}\n{r['sequence']}\n"
                    for r in recs))
    # flat CSV - openable in Sheets straight from Drive while the run continues
    cols = ["design","seed","group","length","seconds","loss","time","sequence"]
    _atomic(f"{out_dir}/progress.csv",
            ",".join(cols) + "\n" +
            "".join(",".join(str(r.get(c,"")) for c in cols) + "\n" for r in recs))
    return len(recs)

interrupted = False
for gname in RUN_GROUPS:
    if interrupted: break
    g   = GROUPS[gname]
    OUT = f"{ROOT}/5G51_{gname}"
    os.makedirs(OUT, exist_ok=True)
    jl   = f"{OUT}/designs.jsonl"
    done = sum(1 for _ in open(jl)) if os.path.exists(jl) else 0
    print(f"\n######## {gname} idx={g['idx']} PDB {g['pdb']} [{done}/{N_DESIGNS} saved] ########", flush=True)
    write_status(state="starting group", group=gname, done_in_group=done, target=N_DESIGNS)
    if done >= N_DESIGNS:
        print("  complete, skipping"); continue

    loss = build_objective(g["idx"])

    for d in range(done, N_DESIGNS):
        seed = int(np.random.randint(100000))
        try:
            x0 = jax.nn.softmax(0.5*jax.random.gumbel(
                key=jax.random.key(seed), shape=(BINDER_LENGTH,20)))
            print(f"\n=== {gname} {d+1}/{N_DESIGNS} (seed {seed}) ===", flush=True)
            t0 = time.time()
            aux, PSSM = simplex_APGM(loss_function=loss, x=x0, n_steps=N_STEPS,
                                     stepsize=0.1, momentum=0.9)
            dt = time.time()-t0
            try:    final_loss = float(np.asarray(aux).ravel()[-1])
            except Exception: final_loss = ""
        except KeyboardInterrupt:
            print("\n!! interrupted - everything finished is saved"); interrupted=True; break
        except Exception:
            print(f"!! design {d+1} failed, continuing:"); traceback.print_exc(); continue

        seq = "".join(TOKENS[int(i)] for i in np.array(PSSM).argmax(-1))
        rec = {"design":d+1,"seed":seed,"group":gname,"epitope_pdb":g["pdb"],
               "epitope_idx":g["idx"],"length":BINDER_LENGTH,"sequence":seq,
               "seconds":round(dt,1),"loss":final_loss,
               "time":datetime.datetime.now().isoformat()}
        n = save_design(OUT, rec, PSSM); print(seq)
        TIMES.append(dt); med = float(np.median(TIMES))
        left = (N_DESIGNS-d-1) + sum(N_DESIGNS for _ in RUN_GROUPS[RUN_GROUPS.index(gname)+1:])
        write_status(state="running", group=gname, done_in_group=n, target=N_DESIGNS,
                     total_done=len(TIMES), remaining=left,
                     median_min=round(med/60,1), eta_hours=round(med*left/3600,1),
                     est_CU=round(med*left/3600*CU_PER_HOUR))
        print(f"    {dt/60:.1f} min | median {med/60:.1f} | saved {n} | "
              f"~{med*left/3600:.1f} h left (~{med*left/3600*CU_PER_HOUR:.0f} CU)")

if TIMES:
    med = float(np.median(TIMES))
    print(f"\nmedian {med/60:.1f} min/design over {len(TIMES)}")
    print(f"  2500 designs would be {med*2500/3600:.0f} h (~{med*2500/3600*CU_PER_HOUR:.0f} CU)")
write_status(state="interrupted" if interrupted else "complete",
             total_done=len(TIMES),
             median_min=round(float(np.median(TIMES))/60,1) if TIMES else None)
print(f"\nlive status: {ROOT}/status.txt   per-group CSV: {ROOT}/5G51_<group>/progress.csv")
print("\nINTERRUPTED - rerun to resume" if interrupted else "\nALL GROUPS COMPLETE")

## 8 · Live monitor — safe to run WHILE the run is going

Reads only from Drive, so run it in a second Colab tab (or re-run this cell) any time to see progress
without touching the running job.

In [ ]:
import json, os, glob
ROOT = globals().get("ROOT", "/content/drive/MyDrive/Mosaic")

st = f"{ROOT}/status.json"
if os.path.exists(st):
    print(open(f"{ROOT}/status.txt").read())
else:
    print("no status yet - the run has not written its first design")

for d in sorted(glob.glob(f"{ROOT}/5G51_*")):
    jl = f"{d}/designs.jsonl"
    n  = sum(1 for _ in open(jl)) if os.path.exists(jl) else 0
    print(f"\n{os.path.basename(d)}: {n} designs saved")
    if n:
        rs = [json.loads(l) for l in open(jl)]
        last = rs[-1]
        print(f"  latest  #{last['design']} seed {last['seed']} "
              f"({last.get('seconds','?')}s)  loss={last.get('loss','')}")
        print(f"  files   {', '.join(sorted(os.path.basename(f) for f in glob.glob(d+'/*') if not f.endswith('.npy')))}")
        print(f"  {last['sequence']}")

## 9 · Results

In [ ]:
import json, collections, os
for gname in RUN_GROUPS:
    f = f"{ROOT}/5G51_{gname}/designs.jsonl"
    if not os.path.exists(f): print(f"{gname}: nothing yet"); continue
    seqs = [json.loads(l) for l in open(f)]
    print(f"\n=== {gname}  PDB {GROUPS[gname]['pdb']}  {len(seqs)} designs ===")
    for r in seqs:
        q = r["sequence"]; c = collections.Counter(q)
        print(f"  #{r['design']:<3} hyd {sum(c[x] for x in 'AVILMFWY')/len(q):.2f}"
              f"  chg {sum(c[x] for x in 'DEKR')/len(q):.2f}  Cys {c['C']}")
        print(f"      {q}")
print("\nNEXT: fold with Boltz-2/AF2 and score ipSAE before ordering.")